In [9]:
import requests
import pandas as pd
import time
import random
from datetime import datetime

In [24]:
ARCTIC_BASE = "https://arctic-shift.photon-reddit.com/api"
SUBREDDITS = ["Ulta", "Sephora", "MUAontheCheap", "MakeupAddiction", "SkincareAddiction"]
POSTS_LIMIT = 100

def fetch_arctic_posts(subreddit, limit=100):
    endpoint = f"{ARCTIC_BASE}/posts/search"
    params = {"subreddit": subreddit, "limit": limit}
    try:
        r = requests.get(endpoint, params=params, timeout=30)
        r.raise_for_status()
        return r.json().get("data", [])
    except requests.exceptions.RequestException as e:
        print(f"  Arctic Shift error for r/{subreddit}: {e}")
        return []

# --- Fetch ---
all_records = []
for sub in SUBREDDITS:
    print(f"Fetching r/{sub}...")
    posts = fetch_arctic_posts(sub, limit=POSTS_LIMIT)
    print(f"  Retrieved {len(posts)} posts.")
    all_records.extend(posts)
    time.sleep(1)

df_arctic = pd.DataFrame(all_records)
print(f"\nRaw DataFrame shape: {df_arctic.shape}")


Fetching r/Ulta...
  Retrieved 100 posts.
Fetching r/Sephora...
  Retrieved 100 posts.
Fetching r/MUAontheCheap...
  Arctic Shift error for r/MUAontheCheap: 422 Client Error: Unprocessable Entity for url: https://arctic-shift.photon-reddit.com/api/posts/search?subreddit=MUAontheCheap&limit=100
  Retrieved 0 posts.
Fetching r/MakeupAddiction...
  Retrieved 100 posts.
Fetching r/SkincareAddiction...
  Retrieved 100 posts.

Raw DataFrame shape: (400, 123)


In [29]:
for col in df_arctic.columns:
    print(f"Column: {col}, Non-null count: {df_arctic[col].notnull().sum() == 0}")

Column: all_awardings, Non-null count: False
Column: allow_live_comments, Non-null count: False
Column: approved_at_utc, Non-null count: True
Column: approved_by, Non-null count: True
Column: archived, Non-null count: False
Column: author, Non-null count: False
Column: author_flair_background_color, Non-null count: False
Column: author_flair_richtext, Non-null count: False
Column: author_flair_template_id, Non-null count: False
Column: author_flair_text, Non-null count: False
Column: author_flair_text_color, Non-null count: False
Column: author_flair_type, Non-null count: False
Column: author_fullname, Non-null count: False
Column: author_is_blocked, Non-null count: False
Column: author_patreon_flair, Non-null count: False
Column: author_premium, Non-null count: False
Column: awarders, Non-null count: False
Column: banned_at_utc, Non-null count: True
Column: banned_by, Non-null count: True
Column: can_gild, Non-null count: False
Column: can_mod_post, Non-null count: False
Column: categ

In [30]:
# --- Normalize (now that we know the real schema) ---
REQUIRED_COLS = ["id", "subreddit", "author", "title", "selftext", "score",
                 "num_comments", "created_utc", "permalink", "url"]

def normalize_df(df, cols=REQUIRED_COLS):
    out = pd.DataFrame(index=df.index)
    for col in cols:
        out[col] = df[col] if col in df.columns else None
    return out

df_clean = normalize_df(df_arctic)

# --- Filter: drop posts with no analyzable text ---
# '[removed]' and '[deleted]' are placeholders, not real content.
BAD_TEXTS = {"[removed]", "[deleted]", None, "", "[ Removed by Reddit ]"}

df_clean["selftext"] = df_clean["selftext"].fillna("")
df_clean["title"]    = df_clean["title"].fillna("")

before = len(df_clean)
df_clean = df_clean[~df_clean["selftext"].isin(BAD_TEXTS) | ~df_clean["title"].isin(BAD_TEXTS)]
df_clean = df_clean[df_clean["title"].str.strip().ne("")]  # require a title
after = len(df_clean)
print(f"Kept {after}/{before} posts with usable text.")

# --- Convert timestamps ---
df_clean["created_datetime"] = pd.to_datetime(df_clean["created_utc"], unit="s", errors="coerce")

print("\nNull counts:")
print(df_clean.isna().sum())
print("\nSubreddit distribution:")
print(df_clean["subreddit"].value_counts())
print("\nSample:")
print(df_clean[["subreddit", "title", "score", "created_datetime"]].head())

Kept 400/400 posts with usable text.

Null counts:
id                  0
subreddit           0
author              0
title               0
selftext            0
score               0
num_comments        0
created_utc         0
permalink           0
url                 0
created_datetime    0
dtype: int64

Subreddit distribution:
subreddit
Ulta                 100
Sephora              100
MakeupAddiction      100
SkincareAddiction    100
Name: count, dtype: int64

Sample:
  subreddit                                              title  score  \
0      Ulta  im so confused how you all are gtting so many ...      1   
1      Ulta      Is Ulta still selling the beauty mini brands?      1   
2      Ulta  I’m so excited for this I’ve been dying to try it      2   
3      Ulta  Sooo...Did anyone receive an Ulta marketing em...      1   
4      Ulta  Do we get a black Friday/ hello Holidays sale ...      1   

     created_datetime  
0 2026-10-17 19:38:25  
1 2026-10-17 17:39:57  
2 2026-10-17 

In [31]:
import os
os.makedirs("../data/raw", exist_ok=True)
df_clean.to_csv("../data/raw/reddit_posts_extracted.csv", index=False)
print(f"Saved {len(df_clean)} rows to ../data/raw/reddit_posts_extracted.csv")

Saved 400 rows to ../data/raw/reddit_posts_extracted.csv


In [36]:
import os
from sqlalchemy import create_engine, types
from dotenv import load_dotenv

load_dotenv()

DB_URL = (
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(DB_URL)

dtype_mapping = {
    "id":               types.VARCHAR(20),
    "subreddit":        types.VARCHAR(50),
    "author":           types.VARCHAR(100),
    "title":            types.TEXT,
    "selftext":         types.TEXT,
    "score":            types.INTEGER,
    "num_comments":     types.INTEGER,
    "created_utc":      types.BIGINT,
    "permalink":        types.TEXT,
    "url":              types.TEXT,
    "created_datetime": types.TIMESTAMP,
}

df_clean.to_sql(
    name="raw_reddit_posts",
    con=engine,
    if_exists="replace",
    index=False,
    dtype=dtype_mapping,
    method="multi",
    chunksize=1000,
)

print(f"Loaded {len(df_clean)} rows into raw_reddit_posts.")
engine.dispose()

Loaded 400 rows into raw_reddit_posts.
